In [3]:
import pandas as pd
import torch
import numpy as np
from numpy.linalg import norm

In [4]:
from transformers import (pipeline, AutoTokenizer, AutoModel)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
xlmr_sent = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

mbert_sent = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/bert-base-multilingual-cased-sentiment-multilingual",
    tokenizer="cardiffnlp/bert-base-multilingual-cased-sentiment-multilingual",
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


In [7]:
pairs = [
  {"id": 1, "archaic": "Leyli bu itabı çün eşitdi, Öz könlündə müqərrər etdi Kim: “Şəbədeyi-sipehri-zalim, Tərh eylədi nəqşi-nəmülaim...”", "modern": "Leyli bu məzəmməti eşidəndə ürəyində belə qərara gəldi ki, zalım fələk onun tale səhnəsində dəhşətli bir oyun çıxarıb...", "gold_label": "negative"},
  {"id": 2, "archaic": "Nacar tutub təriqi-inkar, Əsari-təcaül etdi izhar; Gülzari-itab ab verdi, Giryan-giryan cavab verdi:", "modern": "Əlacsız qəlb danmaq yolunu tutdu. Özünü bilməzliyə vurub məzəmmət gülşənini sulara ağlaya-ağlaya cavab verdi:", "gold_label": "negative"},
  {"id": 3, "archaic": "“Key munisi-ruzigarım ana! Dürrü-düri-şəhvarım ana! Sözlər dersən ki, bilməzəm mən, Məzmumunu fəhm qılmazam mən”", "modern": "“Ey ömür-günümün munisi ana! Mənim şahənə incimin mücrüsü ana! Elə sözlər deyirsən ki, başa düşmürəm, məzmununu anlamıram.”", "gold_label": "neutral"},
  {"id": 4, "archaic": "“Dersən məşuqi eşqü aşiq, Mən sadəzəmir tifli-sadiq; Bilmən, nədir ol hadisə məzmun? Söylə! Necə olmayım digərgün?”", "modern": "“Məşuqə, aşiq, eşq deyirsən! Mən sadədil, sadiq uşağam, onların mənasını bilmirəm. Söylə, mən necə matəəl qalmayım?”", "gold_label": "neutral"},
  {"id": 5, "archaic": "“Eşqin qılmazdı kimsə yadin, Ha səndən eşitdim indi adın. Billah! Nədir, ana, eşqə məfhum? Bu sirri-nihani eylə məlum!”", "modern": "“Heç kəsdən eşq sözünü eşitməmişdim. Bunu indi səndən eşitdim. Allah eşqinə, ana, de görüm, eşqin mənası nədir? Bu gizli sirri aç!”", "gold_label": "neutral"},
  {"id": 6, "archaic": "“Hadiyi-rəhi-muradım olgil! Bu sivədə ustadım olgil! Mən məktəbə rəyim ilə getmən, Bir şüğli xilafi-rəyin etmən.”", "modern": "“Mənim murad yolumda rəhbərim ol, bu işdə ustadım ol. Mən məktəbə özbaşıma getmirəm, sənin fikrinə zidd iş tutmuram.”", "gold_label": "neutral"},
  {"id": 7, "archaic": "“Həm dersən sən ki: ‘Məktəbə var!’ Həm dersən sən ki: ‘Getmə, zinhar!’ Qangı sözə etiqadım olsun? Sənə necə etimadım olsun?”", "modern": "“Həm özün deyirsən ki, ‘məktəbə get!’ Həm də deyirsən ki, ‘ay aman, getmə!’ Hansı sözə inanım? Sənə necə etibar edim?”", "gold_label": "negative"},
  {"id": 8, "archaic": "“...Artıq bu sözü mükərrər etmə, Lütf eylə, məni mükəddər etmə!”", "modern": "...Bu sözü bir daha təkrar eləmə, mənə yazığın gəlsin, dərd salma.", "gold_label": "negative"},
  {"id": 9, "archaic": "Çün ana eşitdi bu cavabi, Tərk etdi şikayətü itabi.", "modern": "Ana bu cavabı eşidib, şikayət və töhmətdən əl çəkdi.", "gold_label": "neutral"},
  {"id": 10, "archaic": "Bihudədir ol qamu fəsanə Kim, aşiğidir filan filanə.", "modern": "Danışılanların hamısının boş əfsanə olduğuna inanıb təsəlli tapdı.", "gold_label": "neutral"},
  {"id": 11, "archaic": "Leyli həm oturdu evdə naçar, Düzdü sədəfinə dürri-şəhvar...", "modern": "Leyli də əlacsız evdə oturdu, şahana inciyə bənzər göz yaşları axıtdı...", "gold_label": "negative"},
  {"id": 12, "archaic": "Fələk ayırdı məni dövr ilə cananımdan,", "modern": "Fələk zülm ilə məni cananımdan ayırdı.", "gold_label": "negative"},
  {"id": 13, "archaic": "Həzər etməzmi əcəb naləvü əfqanımdan?", "modern": "Əcaba, mənim nalə və əfqanımdan cana gəlməzmi?", "gold_label": "negative"},
  {"id": 14, "archaic": "Oda yandırmasa gər şöylə idi noh fələk,", "modern": "Əgər şöylə idi göylərin doqquz qatını yandırmasa,", "gold_label": "neutral"},
  {"id": 15, "archaic": "Nə bitər atəşi-ahi-dili-suzanımdan?", "modern": "mənim yanar ürəyimin ahının odundan nə çıxar?", "gold_label": "negative"},
  {"id": 16, "archaic": "Qəmi-pünhan məni öldürdü, bu həm bir qəm kim,", "modern": "Gizli qəmim məni öldürdü, bu da bir qəm ki,", "gold_label": "negative"},
  {"id": 17, "archaic": "Gülüzrüm olmadı agah qəmi-pünhanımdan.", "modern": "gül üzlümün bu gizli qəmimdən xəbəri olmadı.", "gold_label": "negative"},
  {"id": 18, "archaic": "Lütf edib sən məgər, ey bad, bu gündən böylə,", "modern": "Lütf edib, ey külək, bu gündən belə,", "gold_label": "neutral"},
  {"id": 19, "archaic": "Verəsən bir xəbər ol sərvi-xuramanımdan.", "modern": "bəlkə, sən bir xəbər versən nazlı sərvimdən.", "gold_label": "neutral"},
  {"id": 20, "archaic": "Bir gün ki, bahari-aləmfruz Vermişdi cahanə feyzi-novruz,", "modern": "Bir gün aləmi işıqlandıran bahar dünyaya novruz feyzi gətirmişdi.", "gold_label": "positive"},
  {"id": 21, "archaic": "Bir neçə müsahibi-vəfadar, Məcnuni-şikəsteyi görüb zar", "modern": "Bir neçə vəfalı həmsöhbət Məcnun yazığını görüb hər tərəfdən dedilər ki:", "gold_label": "negative"},
  {"id": 22, "archaic": "Bu fəsldə ədəmi gərək şad, Önduhü bəlayi qəmdən azad!", "modern": "Bu fəsildə gərək adam şad, qəm-qüssədən azad olsun!", "gold_label": "positive"},
  {"id": 23, "archaic": "Məcnuni-həzin ayağa durdu, Səhralara seyr üçün üz urdu.", "modern": "Dərdli Məcnun ayağa durdu, gəzmək üçün səhraya üz tutdu.", "gold_label": "neutral"},
  {"id": 24, "archaic": "Giryən-giryən qılırdı seyran, Heyran-heyən gəzərdi heyran,", "modern": "Ağlaya-ağlaya seyr eləyirdi, heyran-heyran hər yanı gəzirdi,", "gold_label": "negative"},
  {"id": 25, "archaic": "Bir mənzilə düşdü rəhgüzari Kim, seyrdə idi onda yari,", "modern": "Gözlənilmədən yolu bir yerə düşdü ki, orada sevgilisi də bir neçə gülüzlü ilə birlikdə idi.", "gold_label": "neutral"},
  {"id": 26, "archaic": "Bir çəməndə yaşıl çadır qurmuşdu, elə bil ki, ay yaşıllıqda halə salmışdı.", "modern": "Bir çəməndə yaşıl çadır qurmuşdu, elə bil ki, ay yaşıllıqda halə (işıqlı dairə) salmışdı.", "gold_label": "neutral"},
  {"id": 27, "archaic": "Leyli demə – şəm‘i-məclisəfruz, Məcnun demə – atəşi-cigərsuz.", "modern": "Leyli demə – məclisi işıqlandıran bir şam, Məcnun demə – ciyər yandıran atəş.", "gold_label": "positive"},
  {"id": 28, "archaic": "Leyli demə – zülmət içrə bir nur.", "modern": "Leyli demə – zülmət içində nur.", "gold_label": "positive"},
  {"id": 29, "archaic": "Məcnunda qərar tutmayıb huş, Dəryayi-təhəyyür eylədi cuş.", "modern": "Məcnun huşunu itirdi, heyranlıq dəryası coşdu.", "gold_label": "negative"},
  {"id": 30, "archaic": "Leyli də ixtiyarını itirib nigarın bir an belə görə bilmədi.", "modern": "Leyli də ixtiyarını itirib nigarını bir an belə görə bilmədi.", "gold_label": "negative"},
  {"id": 31, "archaic": "Hər yan dedilər ona ki: “Yaxşı baxsan, bu, yamandır, həm sənə, həm də bizə ziyandır”.", "modern": "Hər yandan ona dedilər ki: “Yaxşı baxsan, bu, yamandır, həm sənə, həm də bizə ziyandır”.", "gold_label": "negative"},
  {"id": 32, "archaic": "Söz demədilər bu macəradan, Nə gəncdənü nə əjdahadan.", "modern": "Bu macəradan – nə xəzinədən, nə əjdahadan bir söz açmadılar.", "gold_label": "neutral"},
  {"id": 33, "archaic": "Gördü ki, nigardan nişan yox, Bir cismi-füsürdə var, can yox.", "modern": "Gördü ki, nigarından bir nişan yoxdur, solğun bir bədən var, cansa yoxdur.", "gold_label": "negative"},
  {"id": 34, "archaic": "Həmdərdlərə üz qıldı ağaz: “Key bir neçə həmnişinü həmraz!”", "modern": "Həmdərdlərindən üz istədi: “Ey mənim dostlarım, sirdaşlarım!”,", "gold_label": "neutral"},
  {"id": 35, "archaic": "“Mən rangi-məlamətə boyandım, Sövdazədlik oduna yandım.”", "modern": "“Mən məlamət rənginə boyandım, sevda vurğunluğu odunda yandım.”", "gold_label": "negative"},
  {"id": 36, "archaic": "“Mən bir quşam, uçdum aşiyandan, Mən qandəni meyli-xanədən qandan?”", "modern": "“Mən bir quşam, yuvamdan uçub getdim. Mən hara, ev meyli hara?”", "gold_label": "negative"},
  {"id": 37, "archaic": "“Sən vasiteyi-vücudim oldun! Sən mane‘i-feyzi-cudim oldun.”", "modern": "“Sən mənim yaranmağımın vasitəsi oldun! Yoxluq nəşəsindən məni ayırdın.”", "gold_label": "positive"},
  {"id": 38, "archaic": "“Umdun ki, mənimlə olasan şad, Dərd ki, ümidin oldu bərbad.”", "modern": "“İstədin ki, mənimlə şad olasan, nə böyük dərd ki, ümidin bərbad oldu.”", "gold_label": "negative"},
  {"id": 39, "archaic": "“Mən yox olubam, sən indi var ol! Özə xələfə ümidvar ol!”", "modern": "“Mən yox olmuşam, indi sən var ol! Ümidini özgə oğula bağla.”", "gold_label": "negative"},
  {"id": 40, "archaic": "Söz cövhərinə olan xəridar Bu növ ilə qıldı gərm bazar...", "modern": "Söz cövhərinin alıcısı bazarı belə qızışdırdı: o qoca qəm içində aciz qalıb Məcnunun zəncirlənməsi fikrinə düşdü...", "gold_label": "neutral"},
  {"id": 41, "archaic": "Çün Leyli atası bildi hali, Uyurdu əkabirü əali.", "modern": "Leylinin atası xəbər tutan kimi uca rütbəliləri və böyükləri toplayıb onları qarşılamğa çıxdılar...", "gold_label": "neutral"},
  {"id": 42, "archaic": "Çün şəm sifat olar oturdu, Ol sərv kimi ayağa durdu.", "modern": "Hamı şam kimi oturdu, o isə sərv kimi ayağa qalxdı.", "gold_label": "neutral"},
  {"id": 43, "archaic": "Xanlar götürüldüyündə, ol pir Təqrib ilə dərdin etdi təqrir...", "modern": "Süfrə yığışdırıldıqdan sonra o qoca dərdini tədriclə açdı...", "gold_label": "neutral"},
  {"id": 44, "archaic": "Nəxli-əməl im səmər veribdir, İyzəd mənə bir güvhər veribdir...", "modern": "“Əməlimin xurma ağacı bar verib – tanrı mənə bir gövhər veribdir...”", "gold_label": "positive"},
  {"id": 45, "archaic": "Bir ləlin eşitmişəm sənin var Kim, lölöümə odur sazəvar.", "modern": "Eşitmişəm, sənin də mənim incimə yarasan bir ləlin var...", "gold_label": "positive"},
  {"id": 46, "archaic": "Fəhm et sözümü, təğafül etmə! Bir xeyr işidir, təəllül etmə!", "modern": "Sözümü anla, qəflətdə də qalma, bu xeyir işdir, gecikmə.", "gold_label": "neutral"},
  {"id": 47, "archaic": "Ol sərv-səmənbərin atası... dedi ki: “Ey xirdmənd... Biləmən necə verəyin cavabın?”", "modern": "O sərv boylunun atası... cavab verdi: “...Kitabın (müraciətin) çox çətindir, heç bilmirəm, cavabını necə verim?”", "gold_label": "negative"},
  {"id": 48, "archaic": "Qürbün bilürəm mənə şərəfdir, Əmma xələfin əcəb xələfdir. “Məcnun” – deyə tənə edər xəlaiq...", "modern": "Bilirəm ki, sənin qohumluğun mənə şərəfdir. Ancaq varisin çox qəribə övladdır... Xalq ona “Məcnun” deyə tənə edir.", "gold_label": "negative"},
  {"id": 49, "archaic": "Ol sahibi-nəngü namü namus Döndü evə gəldi xarü mə’yus.", "modern": "O heya sahibi, adlı-sanlı namuslu qoca xar-məyus evinə döndü.", "gold_label": "negative"},
  {"id": 50, "archaic": "Məcnuna dedi ki: “Ey cəfakes! Hacət bitər, olmağı müxaləvəs! Əql ilə açılır ol müəmmə...", "modern": "Məcnuna dedi ki: “Ey cəfakes! Hər dərdin əlacı vardır... Ağıllıların sözünə qulaq as...”", "gold_label": "neutral"},
]


In [8]:
xlmr_name = "xlm-roberta-base"
mbert_name = "bert-base-multilingual-cased"

xlmr_tok = AutoTokenizer.from_pretrained(xlmr_name)
xlmr_emb = AutoModel.from_pretrained(xlmr_name).to(device)

mbert_tok = AutoTokenizer.from_pretrained(mbert_name)
mbert_emb = AutoModel.from_pretrained(mbert_name).to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [9]:
def mean_pooling(output, mask):
    token_embeddings = output.last_hidden_state
    mask = mask.unsqueeze(-1).expand(token_embeddings.size())
    return (token_embeddings * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

def encode(texts, tokenizer, model, batch_size=16):
    model.eval()
    all_emb = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            output = model(**encoded)
            pooled = mean_pooling(output, encoded["attention_mask"])
            all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

In [10]:
archaic_texts = [p["archaic"] for p in pairs]
modern_texts  = [p["modern"] for p in pairs]

xlmr_arch = xlmr_sent(archaic_texts, batch_size=16)
xlmr_mod  = xlmr_sent(modern_texts, batch_size=16)

mbert_arch = mbert_sent(archaic_texts, batch_size=16)
mbert_mod  = mbert_sent(modern_texts, batch_size=16)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [11]:
xlmr_arch_emb = encode(archaic_texts, xlmr_tok, xlmr_emb)
xlmr_mod_emb  = encode(modern_texts,  xlmr_tok, xlmr_emb)

mbert_arch_emb = encode(archaic_texts, mbert_tok, mbert_emb)
mbert_mod_emb  = encode(modern_texts,  mbert_tok, mbert_emb)

In [12]:
rows = []
for i, p in enumerate(pairs):
    rows.append({
        "text_archaic": p["archaic"],
        "text_modern": p["modern"],
        "gold_label": p["gold_label"],

        "xlmr_arch_label": xlmr_arch[i]["label"],
        "xlmr_mod_label":  xlmr_mod[i]["label"],
        "mbert_arch_label": mbert_arch[i]["label"],
        "mbert_mod_label":  mbert_mod[i]["label"],

        "xlmr_similarity": cosine_sim(xlmr_arch_emb[i], xlmr_mod_emb[i]),
        "mbert_similarity": cosine_sim(mbert_arch_emb[i], mbert_mod_emb[i]),
    })

df = pd.DataFrame(rows)

In [13]:
df

,text_archaic,text_modern,gold_label,xlmr_arch_label,xlmr_mod_label,mbert_arch_label,mbert_mod_label,xlmr_similarity,mbert_similarity
0,"Leyli bu itabı çün eşitdi, Öz könlündə müqərrə...",Leyli bu məzəmməti eşidəndə ürəyində belə qəra...,negative,positive,neutral,negative,negative,0.998164,0.857831
1,"Nacar tutub təriqi-inkar, Əsari-təcaül etdi iz...",Əlacsız qəlb danmaq yolunu tutdu. Özünü bilməz...,negative,neutral,negative,negative,negative,0.998184,0.809717
2,“Key munisi-ruzigarım ana! Dürrü-düri-şəhvarım...,“Ey ömür-günümün munisi ana! Mənim şahənə inci...,neutral,neutral,positive,negative,positive,0.998931,0.903329
3,"“Dersən məşuqi eşqü aşiq, Mən sadəzəmir tifli-...","“Məşuqə, aşiq, eşq deyirsən! Mən sadədil, sadi...",neutral,neutral,neutral,negative,negative,0.999306,0.904952
4,"“Eşqin qılmazdı kimsə yadin, Ha səndən eşitdim...",“Heç kəsdən eşq sözünü eşitməmişdim. Bunu indi...,neutral,positive,negative,negative,negative,0.998551,0.903436
5,“Hadiyi-rəhi-muradım olgil! Bu sivədə ustadım ...,"“Mənim murad yolumda rəhbərim ol, bu işdə usta...",neutral,negative,neutral,positive,negative,0.998457,0.899839
6,“Həm dersən sən ki: ‘Məktəbə var!’ Həm dersən ...,"“Həm özün deyirsən ki, ‘məktəbə get!’ Həm də d...",negative,neutral,neutral,neutral,negative,0.998876,0.952069
7,"“...Artıq bu sözü mükərrər etmə, Lütf eylə, mə...","...Bu sözü bir daha təkrar eləmə, mənə yazığın...",negative,positive,negative,positive,negative,0.997733,0.802954
8,"Çün ana eşitdi bu cavabi, Tərk etdi şikayətü i...","Ana bu cavabı eşidib, şikayət və töhmətdən əl ...",neutral,negative,negative,negative,negative,0.997276,0.840987
9,"Bihudədir ol qamu fəsanə Kim, aşiğidir filan f...",Danışılanların hamısının boş əfsanə olduğuna i...,neutral,neutral,negative,negative,negative,0.994906,0.678515


In [14]:
df["xlmr_arch_correct"]  = df["xlmr_arch_label"]  == df["gold_label"]
df["xlmr_mod_correct"]   = df["xlmr_mod_label"]   == df["gold_label"]
df["mbert_arch_correct"] = df["mbert_arch_label"] == df["gold_label"]
df["mbert_mod_correct"]  = df["mbert_mod_label"]  == df["gold_label"]

df["xlmr_consistent"]  = df["xlmr_arch_label"]  == df["xlmr_mod_label"]
df["mbert_consistent"] = df["mbert_arch_label"] == df["mbert_mod_label"]

print("\n=== ACCURACY ===")
print("XLM-R archaic:", df["xlmr_arch_correct"].mean())
print("XLM-R modern :", df["xlmr_mod_correct"].mean())
print("mBERT archaic:", df["mbert_arch_correct"].mean())
print("mBERT modern :", df["mbert_mod_correct"].mean())

print("\n=== CONSISTENCY ===")
print("XLM-R:", df["xlmr_consistent"].mean())
print("mBERT:", df["mbert_consistent"].mean())


=== ACCURACY ===
XLM-R archaic: 0.44
XLM-R modern : 0.52
mBERT archaic: 0.36
mBERT modern : 0.34

=== CONSISTENCY ===
XLM-R: 0.54
mBERT: 0.72


In [15]:
# save results to excel

from openpyxl import load_workbook
from openpyxl.styles import PatternFill

excel_path = "fuzuli_sentiment_similarity_colored.xlsx"
df.to_excel(excel_path, index=False)

wb = load_workbook(excel_path)
ws = wb.active

GREEN = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
RED = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
YELLOW = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")

DARK_GREEN = PatternFill(start_color="63BE7B", end_color="63BE7B", fill_type="solid")
LIGHT_GREEN = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
ORANGE = PatternFill(start_color="F4B084", end_color="F4B084", fill_type="solid")

header = {cell.value: idx + 1 for idx, cell in enumerate(ws[1])}

gold_col = header["gold_label"]

xlmr_arch_col = header["xlmr_arch_label"]
xlmr_mod_col = header["xlmr_mod_label"]

mbert_arch_col = header["mbert_arch_label"]
mbert_mod_col = header["mbert_mod_label"]

xlmr_sim_col = header["xlmr_similarity"]
mbert_sim_col = header["mbert_similarity"]

for row in range(2, ws.max_row + 1):

    gold = ws.cell(row, gold_col).value

    for col in [xlmr_arch_col, xlmr_mod_col, mbert_arch_col, mbert_mod_col]:
        cell = ws.cell(row, col)
        if cell.value == gold:
            cell.fill = GREEN
        else:
            cell.fill = RED

    if ws.cell(row, xlmr_arch_col).value == ws.cell(row, xlmr_mod_col).value:
        ws.cell(row, xlmr_mod_col).fill = GREEN
    else:
        ws.cell(row, xlmr_mod_col).fill = YELLOW

    if ws.cell(row, mbert_arch_col).value == ws.cell(row, mbert_mod_col).value:
        ws.cell(row, mbert_mod_col).fill = GREEN
    else:
        ws.cell(row, mbert_mod_col).fill = YELLOW

    for col in [xlmr_sim_col, mbert_sim_col]:
        sim_cell = ws.cell(row, col)
        sim = float(sim_cell.value)

        if sim >= 0.75:
            sim_cell.fill = DARK_GREEN
        elif sim >= 0.60:
            sim_cell.fill = LIGHT_GREEN
        elif sim >= 0.45:
            sim_cell.fill = ORANGE
        else:
            sim_cell.fill = RED

wb.save(excel_path)
print(f"Saved color-coded file: {excel_path}")


Saved color-coded file: fuzuli_sentiment_similarity_colored.xlsx


In [16]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

labels = ["negative", "neutral", "positive"]

#xlmr metrics
print("\n=== XLM-R (ARCHAIC) ===")
print(classification_report(
    df["gold_label"],
    df["xlmr_arch_label"],
    labels=labels,
    digits=4
))

print("\n=== XLM-R (MODERN) ===")
print(classification_report(
    df["gold_label"],
    df["xlmr_mod_label"],
    labels=labels,
    digits=4
))

#mbert metrics
print("\n=== mBERT (ARCHAIC) ===")
print(classification_report(
    df["gold_label"],
    df["mbert_arch_label"],
    labels=labels,
    digits=4
))

print("\n=== mBERT (MODERN) ===")
print(classification_report(
    df["gold_label"],
    df["mbert_mod_label"],
    labels=labels,
    digits=4
))



=== XLM-R (ARCHAIC) ===
              precision    recall  f1-score   support

    negative     0.5789    0.4783    0.5238        23
     neutral     0.4348    0.5000    0.4651        20
    positive     0.1250    0.1429    0.1333         7

    accuracy                         0.4400        50
   macro avg     0.3796    0.3737    0.3741        50
weighted avg     0.4577    0.4400    0.4457        50


=== XLM-R (MODERN) ===
              precision    recall  f1-score   support

    negative     0.6250    0.6522    0.6383        23
     neutral     0.4667    0.3500    0.4000        20
    positive     0.3636    0.5714    0.4444         7

    accuracy                         0.5200        50
   macro avg     0.4851    0.5245    0.4942        50
weighted avg     0.5251    0.5200    0.5158        50


=== mBERT (ARCHAIC) ===
              precision    recall  f1-score   support

    negative     0.4048    0.7391    0.5231        23
     neutral     0.0000    0.0000    0.0000        20
 

In [17]:
def prf(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro"
    )
    return p, r, f

metrics = []

metrics.append(("XLM-R", "archaic", *prf(df["gold_label"], df["xlmr_arch_label"])))
metrics.append(("XLM-R", "modern",  *prf(df["gold_label"], df["xlmr_mod_label"])))
metrics.append(("mBERT", "archaic", *prf(df["gold_label"], df["mbert_arch_label"])))
metrics.append(("mBERT", "modern",  *prf(df["gold_label"], df["mbert_mod_label"])))

metrics_df = pd.DataFrame(
    metrics,
    columns=["model", "text_type", "precision", "recall", "f1"]
)

metrics_df.to_csv("fuzuli_prf_metrics.csv", index=False)
print(metrics_df)


   model text_type  precision    recall        f1
0  XLM-R   archaic   0.379577  0.373706  0.374086
1  XLM-R    modern   0.485101  0.524534  0.494247
2  mBERT   archaic   0.182540  0.293996  0.221978
3  mBERT    modern   0.192308  0.279503  0.223325
